# 01 — Data Cleaning: Food.com + RecipeNLG

Goal: clean both datasets independently first, confirm schemas line up, THEN merge.
Do not merge blind — Food.com has nutrition data, RecipeNLG does not.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import ast
import numpy as np

pd.set_option('display.max_colwidth', 80)

## 1. Load raw datasets

Set `nrows=None` once you're confident this runs clean end-to-end. Start capped for iteration speed.

In [ ]:
RAW_DIR = Path.cwd().parent / "datasets" / "raw"

NROWS = 50000  # set to None for full dataset once pipeline is validated

recipes = pd.read_csv(RAW_DIR / "RAW_recipes.csv", nrows=NROWS)
recipenlg = pd.read_csv(RAW_DIR / "full_dataset.csv", nrows=NROWS)

print("Food.com:", recipes.shape)
print("RecipeNLG:", recipenlg.shape)

## 2. Clean Food.com dataset

Key issue: `nutrition`, `tags`, `steps`, `ingredients` are all stringified Python lists (read from CSV as strings, not actual lists). Must be parsed with `ast.literal_eval`, not `eval` (safety) and not `.split(',')` (breaks on nested commas inside values).

Nutrition list order per Food.com's documentation: `[calories, total_fat_PDV, sugar_PDV, sodium_PDV, protein_PDV, saturated_fat_PDV, carbohydrates_PDV]`. Everything except calories is **% Daily Value**, not grams — do not treat these as raw gram quantities downstream.

In [ ]:
def safe_literal_eval(val):
    """Parse stringified list columns safely. Returns empty list on failure."""
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return []


def clean_food_com(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Drop rows with no name or no ingredients — unusable for recommendation
    df = df.dropna(subset=["name", "ingredients"])

    # Parse stringified list columns
    df["tags"] = df["tags"].apply(safe_literal_eval)
    df["steps"] = df["steps"].apply(safe_literal_eval)
    df["ingredients"] = df["ingredients"].apply(safe_literal_eval)
    df["nutrition"] = df["nutrition"].apply(safe_literal_eval)

    # Expand nutrition list into named columns
    nutrition_cols = ["calories", "total_fat_pdv", "sugar_pdv", "sodium_pdv",
                       "protein_pdv", "sat_fat_pdv", "carbs_pdv"]
    nutrition_expanded = pd.DataFrame(
        df["nutrition"].tolist(), index=df.index, columns=nutrition_cols
    )
    df = pd.concat([df, nutrition_expanded], axis=1)

    # Drop obviously broken rows: 0 ingredients, 0 steps, or absurd calorie outliers
    df = df[df["ingredients"].map(len) > 0]
    df = df[df["steps"].map(len) > 0]
    df = df[(df["calories"] > 0) & (df["calories"] < 5000)]  # >5000 cal/serving is almost certainly a data error

    # Fill description NaNs — don't drop rows just for missing description
    df["description"] = df["description"].fillna("")

    # Normalize text fields
    df["name"] = df["name"].str.strip().str.lower()

    df["source"] = "food.com"

    return df.reset_index(drop=True)


recipes_clean = clean_food_com(recipes)
print(f"Food.com: {len(recipes)} -> {len(recipes_clean)} rows after cleaning")
recipes_clean[["name", "ingredients", "calories", "n_ingredients", "n_steps", "source"]].head(3)

## 3. Clean RecipeNLG dataset

No nutrition data here — that's a permanent asymmetry, not something to paper over. `NER` is already a clean parsed ingredient list (better quality than raw `ingredients`, which has quantities/units mixed in), so we use `NER` as the canonical ingredient field for this source.

In [ ]:
def clean_recipenlg(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df = df.dropna(subset=["title", "ingredients"])

    df["ingredients"] = df["ingredients"].apply(safe_literal_eval)
    df["directions"] = df["directions"].apply(safe_literal_eval)
    df["NER"] = df["NER"].apply(safe_literal_eval)

    df = df[df["ingredients"].map(len) > 0]
    df = df[df["directions"].map(len) > 0]

    df["title"] = df["title"].str.strip().str.lower()

    df["source"] = "recipenlg"

    return df.reset_index(drop=True)


recipenlg_clean = clean_recipenlg(recipenlg)
print(f"RecipeNLG: {len(recipenlg)} -> {len(recipenlg_clean)} rows after cleaning")
recipenlg_clean[["title", "ingredients", "NER", "source"]].head(3)

## 4. Visualize before merging

Check distributions and missingness on both cleaned sets before committing to a unified schema.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(recipes_clean["calories"], bins=50)
axes[0, 0].set_title("Food.com: Calories distribution")
axes[0, 0].set_xlabel("calories")

axes[0, 1].hist(recipes_clean["n_ingredients"], bins=30)
axes[0, 1].set_title("Food.com: # Ingredients")

axes[1, 0].hist(recipenlg_clean["ingredients"].map(len), bins=30)
axes[1, 0].set_title("RecipeNLG: # Ingredients")

axes[1, 1].hist(recipes_clean["n_steps"], bins=30)
axes[1, 1].set_title("Food.com: # Steps")

plt.tight_layout()
plt.show()

In [ ]:
print("Food.com missingness after cleaning:")
print(recipes_clean.isnull().sum()[recipes_clean.isnull().sum() > 0])
print("\nRecipeNLG missingness after cleaning:")
print(recipenlg_clean.isnull().sum()[recipenlg_clean.isnull().sum() > 0])

print(f"\nFood.com duplicate names: {recipes_clean['name'].duplicated().sum()}")
print(f"RecipeNLG duplicate titles: {recipenlg_clean['title'].duplicated().sum()}")

## 5. Decide unified schema (before merging)

Proposed common columns: `title`, `ingredients` (list), `instructions` (list), `source`, `nutrition_available` (bool), `calories`/`total_fat_pdv`/etc. (NaN for RecipeNLG rows).

**Do not drop RecipeNLG rows for missing nutrition** — that would throw away most of your dataset. Instead keep a `nutrition_available` flag so downstream feature engineering can branch on it (e.g., nutrition-based filtering only applies to Food.com rows, or nutrition gets imputed/estimated later as a separate task).

In [ ]:
nutrition_cols = ["calories", "total_fat_pdv", "sugar_pdv", "sodium_pdv",
                   "protein_pdv", "sat_fat_pdv", "carbs_pdv"]

food_unified = pd.DataFrame({
    "title": recipes_clean["name"],
    "ingredients": recipes_clean["ingredients"],
    "instructions": recipes_clean["steps"],
    "description": recipes_clean["description"],
    "source": recipes_clean["source"],
    "nutrition_available": True,
})
for col in nutrition_cols:
    food_unified[col] = recipes_clean[col]

recipenlg_unified = pd.DataFrame({
    "title": recipenlg_clean["title"],
    "ingredients": recipenlg_clean["NER"],       # cleaner than raw ingredients field
    "instructions": recipenlg_clean["directions"],
    "description": "",
    "source": recipenlg_clean["source"],
    "nutrition_available": False,
})
for col in nutrition_cols:
    recipenlg_unified[col] = np.nan

print("Food.com unified:", food_unified.shape)
print("RecipeNLG unified:", recipenlg_unified.shape)
print("\nColumns match:", list(food_unified.columns) == list(recipenlg_unified.columns))

## 6. Merge into unified dataset

Deduplicate by normalized title across sources — same recipe name appearing in both datasets should not be double-counted at inference time later.

In [ ]:
unified = pd.concat([food_unified, recipenlg_unified], ignore_index=True)

before = len(unified)
unified = unified.drop_duplicates(subset=["title"], keep="first")
print(f"Merged: {before} rows -> {len(unified)} after cross-source title dedup")

print(unified["source"].value_counts())
print(f"\nNutrition available: {unified['nutrition_available'].sum()} / {len(unified)} "
      f"({unified['nutrition_available'].mean()*100:.1f}%)")

In [ ]:
PROCESSED_DIR = Path.cwd().parent / "datasets" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

unified.to_parquet(PROCESSED_DIR / "unified_recipes.parquet", index=False)
print(f"Saved {len(unified)} rows to {PROCESSED_DIR / 'unified_recipes.parquet'}")

## Next notebook: `02_nlp_preprocessing.ipynb`

Before moving on, check:
- Does the ~X% nutrition coverage above match what you expected? If RecipeNLG dominates row count, most of your dataset has no nutrition signal — worth deciding now whether nutrition-based features get used as a hard filter (bad, excludes most rows) or a soft optional feature (better).
- Any titles that look like near-duplicates but weren't caught by exact-match dedup (e.g. "chicken curry" vs "chicken curry recipe") — worth a fuzzy-dedup pass if this shows up a lot.
- Saved as parquet, not CSV — preserves list dtypes (ingredients/instructions) without re-parsing strings every load.